# Explore news-data for issue #6

Profile the source data before sampling: volume, quality, time range, channels, languages, and whether X posts are present.

- Issue: [#6](https://github.com/news-trackers/nlp-workspace/issues/6) (parent [#5](https://github.com/news-trackers/nlp-workspace/issues/5))
- Data: [`news-trackers/news-data`](https://huggingface.co/datasets/news-trackers/news-data) (private), `data/train.parquet`

> **Privacy:** this repo is public. Print aggregates only, never raw post text.

## 0. Setup

Add a Hugging Face **Read** token as a Colab secret named `HF_TOKEN` (🔑 icon, enable *Notebook access*).

## 1. Load

In [1]:
import re
import pandas as pd
from huggingface_hub import hf_hub_download
from google.colab import userdata

REPO_ID = "news-trackers/news-data"
REVISION = "7dee9570f394f6b945a9e2b689888feaf4777962"  # pinned: pr/01 (#1) on main

path = hf_hub_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    filename="data/train.parquet",
    revision=REVISION,
    token=userdata.get("HF_TOKEN"),
)
df = pd.read_parquet(path)

print("rows:", len(df))
print("columns:", list(df.columns))
df.dtypes

data/train.parquet: reconstructing file:   0%|          |  0.00B / 2.27MB            

data/train.parquet: downloading bytes:           |  0.00B            

rows: 8469
columns: ['content', 'post_creation_date', 'insert_date']


,0
content,object
post_creation_date,datetime64[us]
insert_date,datetime64[us]


## 2. Quality

In [2]:
content = df["content"].fillna("")

pd.Series({
    "rows": len(df),
    "null content": int(df["content"].isna().sum()),
    "whitespace-only content": int((content.str.strip() == "").sum()),
    "exact duplicates": int(content.duplicated().sum()),
})

,0
rows,8469
null content,0
whitespace-only content,133
exact duplicates,195


## 3. Time range and crawl delay

Crawl delay = `insert_date` − `post_creation_date`.

In [3]:
for col in ["post_creation_date", "insert_date"]:
    df[col] = pd.to_datetime(df[col], errors="coerce", utc=True)

print(df[["post_creation_date", "insert_date"]].agg(["min", "max"]))
print("\ninvalid dates:", df[["post_creation_date", "insert_date"]].isna().sum().to_dict())

delay_min = (df["insert_date"] - df["post_creation_date"]).dt.total_seconds() / 60
print("\ncrawl delay (minutes):")
print(delay_min.describe(percentiles=[0.5, 0.9, 0.95]).round(1))
print("negative delays:", int((delay_min < 0).sum()))

           post_creation_date               insert_date
min 2026-08-14 10:31:49+00:00 2026-08-14 11:00:02+00:00
max 2026-08-15 06:01:49+00:00 2026-08-15 06:02:46+00:00

invalid dates: {'post_creation_date': 0, 'insert_date': 0}

crawl delay (minutes):
count    8469.0
mean       72.7
std        49.0
min         0.0
50%        65.2
90%       144.4
95%       162.5
max       507.9
dtype: float64
negative delays: 0


In [4]:
posts_per_day = df.set_index("post_creation_date").resample("D").size()
posts_per_day[posts_per_day > 0]

,0
post_creation_date,
2026-08-14 00:00:00+00:00,7234
2026-08-15 00:00:00+00:00,1235


## 4. Channels

There is no channel column. No post starts with a handle (first run), so we also check for a channel signature at the end and for handles anywhere in the text (which may be mentions, not the source).

In [5]:
HANDLE = r"@([A-Za-z0-9_]{5,32})"
stripped = content.str.strip()

start_handle = stripped.str.extract(r"^" + HANDLE, expand=False)
end_handle = stripped.str.extract(HANDLE + r"\W*$", expand=False)
any_handle = content.str.contains(HANDLE, regex=True)

print("handle anywhere:", int(any_handle.sum()))
print("handle at start:", int(start_handle.notna().sum()))
print("handle at end (signature):", int(end_handle.notna().sum()))

df["channel"] = end_handle.fillna(start_handle)
channel_counts = df["channel"].value_counts()
print("\ndistinct channels:", df["channel"].nunique())
channel_counts.head(30)

handle anywhere: 5287
handle at start: 0
handle at end (signature): 4393

distinct channels: 175


/tmp/ipykernel_2274/1485433648.py:6: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  any_handle = content.str.contains(HANDLE, regex=True)


,count
channel,
irannewspaper,182
IRNA_1313,171
EtemadOnline,157
alalamarabic,146
HamshahriNews,137
snntv,131
bornanews_ir,125
TasnimNews,120
AnaNews,111


In [6]:
# most frequent handles anywhere in the text
content.str.findall(HANDLE).explode().dropna().value_counts().head(20)

,count
content,
chandsanieh_news,212
irannewspaper,182
IRNA_1313,172
EtemadOnline,157
alalamarabic,146
AkhbareFori,145
HamshahriNews,137
snntv,131
bornanews_ir,125


## 5. Clean text

Remove the leading handle and trailing signature so channel names don't leak into model input. Mid-text mentions are kept.

In [7]:
text = content.str.replace(r"^\s*@[A-Za-z0-9_]{5,32}\s*[:\-–|]?\s*", "", regex=True)
text = text.str.replace(r"[\s\W]*@[A-Za-z0-9_]{5,32}\W*$", "", regex=True)
df["text"] = text.str.strip()

print("posts changed:", int((df["text"] != content.str.strip()).sum()))
print("empty after cleaning:", int((df["text"] == "").sum()))

posts changed: 4393
empty after cleaning: 134


## 6. Language (rough)

Script-based heuristic; Persian vs Arabic is decided by script-specific letters. For profiling only, not labeling.

In [8]:
def script_label(s: str) -> str:
    counts = {
        "arabic_script": len(re.findall(r"[\u0600-\u06FF]", s)),
        "latin": len(re.findall(r"[A-Za-z]", s)),
        "cyrillic": len(re.findall(r"[\u0400-\u04FF]", s)),
        "hebrew": len(re.findall(r"[\u0590-\u05FF]", s)),
    }
    total = sum(counts.values())
    if total == 0:
        return "no_letters"
    top, n = max(counts.items(), key=lambda kv: kv[1])
    if n / total < 0.8:
        return "mixed"
    if top == "arabic_script":
        persian = len(re.findall(r"[پچژگکی]", s))
        arabic = len(re.findall(r"[كيةى]", s))
        return "fa" if persian >= arabic else "ar"
    return {"latin": "latin", "cyrillic": "cyrillic", "hebrew": "he"}[top]


df["lang_guess"] = df["text"].map(script_label)
df["lang_guess"].value_counts()

,count
lang_guess,
fa,5120
ar,1749
mixed,775
latin,431
cyrillic,256
no_letters,135
he,3


In [9]:
# language by top 15 channels
top_channels = channel_counts.index[:15]
if len(top_channels) == 0:
    print("no channel signatures found; skipping channel x language table")
else:
    display(pd.crosstab(df.loc[df["channel"].isin(top_channels), "channel"], df["lang_guess"]))

lang_guess,ar,fa,mixed
channel,,,
AnaNews,0,111,0
EtemadOnline,0,157,0
FarhikhteganOnline,0,69,18
HamshahriNews,0,137,0
IRNA_1313,2,167,2
MyAsriran,0,79,7
Nournews_IR,0,84,15
Nournews_ir,0,102,0
TasnimNews,0,120,0


## 7. X posts?

Issue #6 asks for Telegram **and** X, but there is no source column; links are the only signal.

In [10]:
x_links = content.str.contains(r"(?:twitter\.com|(?<![\w.])x\.com)/", case=False, regex=True)
tme_links = content.str.contains(r"t\.me/", case=False, regex=True)

print("posts with X/Twitter links:", int(x_links.sum()))
print("posts with t.me links:", int(tme_links.sum()))
print("posts with a channel handle:", int(df["channel"].notna().sum()))

posts with X/Twitter links: 5
posts with t.me links: 65
posts with a channel handle: 4393


## 8. Text length

In [11]:
df["n_chars"] = df["text"].str.len()
df["n_words"] = df["text"].str.split().str.len()

print(df[["n_chars", "n_words"]].describe(percentiles=[0.05, 0.5, 0.95]).round(0))
print("\nposts under 5 words:", int((df["n_words"] < 5).sum()))

       n_chars  n_words
count   8469.0   8469.0
mean     290.0     50.0
std      323.0     57.0
min        0.0      0.0
5%        46.0      8.0
50%      179.0     31.0
95%      843.0    149.0
max     4072.0    781.0

posts under 5 words: 208


## 9. Summary

In [12]:
pd.Series({
    "posts": len(df),
    "published from": str(df["post_creation_date"].min()),
    "published to": str(df["post_creation_date"].max()),
    "distinct channels": df["channel"].nunique(),
    "posts without channel": int(df["channel"].isna().sum()),
    "whitespace-only": int((content.str.strip() == "").sum()),
    "exact duplicates": int(content.duplicated().sum()),
    "under 5 words": int((df["n_words"] < 5).sum()),
    "X links": int(x_links.sum()),
    "t.me links": int(tme_links.sum()),
    "languages (rough)": df["lang_guess"].value_counts().to_dict(),
})

,0
posts,8469
published from,2026-08-14 10:31:49+00:00
published to,2026-08-15 06:01:49+00:00
distinct channels,175
posts without channel,4076
whitespace-only,133
exact duplicates,195
under 5 words,208
X links,5
t.me links,65


## 10. Pilot sample

Random 50 posts to estimate entity density and sentiment class balance for #7. Final stratification is decided in #7.

> Saved in Colab only. Do **not** commit it; it belongs in the private HF dataset.

In [13]:
PILOT_SIZE = 50
SEED = 42

pool = df[(df["n_words"] >= 5) & ~df["text"].duplicated()]
pilot = pool.sample(n=PILOT_SIZE, random_state=SEED)
pilot = pilot.reset_index(names="row_id")[["row_id", "channel", "lang_guess", "post_creation_date", "text"]]

pilot.to_parquet("/content/pilot_sample.parquet", index=False)

print("pilot saved:", len(pilot), "posts")
print(pilot["lang_guess"].value_counts().to_dict())
print(pilot["channel"].value_counts().head(10).to_dict())

pilot saved: 50 posts
{'fa': 24, 'ar': 13, 'mixed': 12, 'latin': 1}
{'EtemadOnline': 3, 'AnaNews': 2, 'Saberinfa': 2, 'iribnews': 2, 'jjonews': 1, 'varzesh3': 1, 'mazandaraniribnews': 1, 'sepah_pasdaran': 1, 'gunaztv2004': 1, 'HamshahriNews': 1}
